# claudeknows: make the gold set (once)

The **gold set** is the model's fixed final exam: photos from your own videos where **you** have
checked the boxes. It is never trained on and never changes, so every new model takes the same
exam. The learning loop may only replace your model with a new one that does better on it.

**What this notebook does**, in one **Runtime -> Run all**:
1. brings in your own videos from Drive `claudeknows_photos/` and boxes them automatically;
2. picks a few **whole videos** (each from a different place) and shows you photos from them;
3. you click **Yes** if every bottle, set of keys and shoe in the photo has a correct box and
   none is missing, **No** otherwise. Each click is saved to Drive straight away;
4. when you are done, run the last cell (step 5) again: it freezes the Yes photos as the gold
   set in Drive `claudeknows_models/gold/`.

The videos it picks are kept out of training from then on, so it never takes the only video
of an object. The loop needs gold photos from **5 different places**; if you don't have that
many videos yet, it tells you what to film.

**Before you start:** Runtime -> Change runtime type -> GPU (e.g. T4). Nothing here is uploaded to GitHub.

In [ ]:
import torch
print("GPU available:", torch.cuda.is_available())

## 1. Get the code

In [ ]:
REPO = "https://github.com/alexg-sketch/claudeknows"
BRANCH = "main"  # change to a claude/... branch to try unmerged work
import os
if not os.path.isdir("claudeknows_repo"):
    !git clone -b $BRANCH $REPO claudeknows_repo
%cd claudeknows_repo
!git fetch -q origin $BRANCH && git reset -q --hard origin/$BRANCH  # always the latest code
!git log -1 --format='Code version: %h %s'
!pip install -q -r requirements.txt transformers

## 2. Connect Google Drive

When the popup appears, click **Connect** and tick **every** box.

In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as e:
    print("Could not connect Google Drive:", e)
if not os.path.isdir("/content/drive/MyDrive"):
    raise SystemExit("Google Drive is NOT connected. Run this cell again, click 'Connect to Google "
                     "Drive' and tick every box.")
PHOTOS = "/content/drive/MyDrive/claudeknows_photos"
MODELS = "/content/drive/MyDrive/claudeknows_models"
GOLD = f"{MODELS}/gold"
ANSWERS = f"{MODELS}/gold_answers.json"  # your clicks, kept if Colab disconnects
if not os.path.isdir(PHOTOS):
    raise SystemExit("No claudeknows_photos folder in My Drive: the gold set is made from your own videos.")
if os.path.isfile(f"{GOLD}/gold.json"):
    raise SystemExit("You already have a gold set (claudeknows_models/gold/). It is frozen on purpose, "
                     "so there is nothing to do here.")
os.makedirs(MODELS, exist_ok=True)
print("Drive connected. Your folders:", sorted(os.listdir(PHOTOS)))

## 3. Bring in your videos and box them

Turns your videos into photos (about 2 per second) and lets the auto-boxer (OWLv2) draw boxes.
Takes a few minutes. Internet photos are not needed here.

In [ ]:
!python -m claudeknows.detection_data import "$PHOTOS" --root data/detection
!python -m claudeknows.autobox run data/detection

## 4. Your questions

Click **Yes** only if **every** water bottle, set of keys and shoe in the photo has a box that fits
it, with the right name, and **none is missing**. Otherwise click **No**. A wrong Yes would put a
mistake in the exam for good, so when unsure, click No. Answers are saved to Drive at once.

In [ ]:
import io
import ipywidgets as W
from IPython.display import display
from claudeknows.detection_data import scan_detection, scene_groups
from claudeknows import gold as G

samples, _ = scan_detection("data/detection")
scenes = scene_groups(samples)
cands = G.pick_gold_scenes(samples, scenes)
answers = G.load_answers(ANSWERS)

def gold_question(s):
    buf = io.BytesIO()
    G.gold_view(s).save(buf, "JPEG")
    yes, no = W.Button(description="Yes"), W.Button(description="No")
    saved = W.Label("" if s.path.name not in answers else
                    "saved: " + ("yes" if answers[s.path.name] else "no"))
    def click(ok):
        G.save_answer(ANSWERS, s.path.name, ok)  # saved to Drive now
        saved.value = "saved: " + ("yes" if ok else "no") + " (click the other button to change it)"
    yes.on_click(lambda _b: click(True))
    no.on_click(lambda _b: click(False))
    return W.VBox([W.Image(value=buf.getvalue(), format="jpeg"),
                   W.Label(f"{s.path.name}: are ALL objects boxed correctly, none missing?"),
                   W.HBox([yes, no, saved])])

for s in cands:
    display(gold_question(s))

## 5. Freeze the gold set

**Run this cell again after answering** (click it, then press Shift+Enter). With "Run all" it runs
before you have answered and just says so. Once frozen, the gold set never changes.
`ALLOW_SMALL = True` freezes it even with fewer than 5 places, but then the loop still can't
upgrade your model, so it's better to film more videos first.

In [ ]:
ALLOW_SMALL = False
manifest = G.freeze_from_answers(cands, G.load_answers(ANSWERS), "data/detection", GOLD,
                                 scenes=scenes, allow_small=ALLOW_SMALL)
if manifest:
    print("Done. Next: run the spatial notebook once, so your current model is trained WITHOUT "
          "the gold videos (it finds the gold set in Drive by itself). Then use the learning loop.")